# Edima — entraînement M2M100 Duala

Kaggle : GPU **T4 ×2**, Internet **ON**, ~30 h/semaine.

Ordre : téléchargement → préparation → token `__dua__` → paires FR synthétiques → entraînement → évaluation → export int8.

Le glossaire validé reste la source de vérité ; le modèle ne sert qu'au texte inédit.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
# Cloner le repo si nécessaire :
# !git clone -b feature/moteur-de-traduction https://github.com/HonluePetnou/edima.git
from pathlib import Path
import os

cwd = Path.cwd()
candidates = [cwd / "edima", cwd, *cwd.parents]
repo = next((p for p in candidates if (p / "ml" / "scripts").is_dir()), None)
if repo is None:
    raise SystemExit("Repo root not found; clone edima and open this notebook from inside it.")
os.chdir(repo)
print("Repo root:", Path.cwd())

In [ ]:
# Sur Kaggle, torch est déjà installé : ne pas le réinstaller.
!pip install -q "transformers>=4.46" datasets accelerate sentencepiece sacremoses sacrebleu ctranslate2 huggingface_hub pyyaml tqdm peft
import transformers, sentencepiece
print("transformers", transformers.__version__, "| sentencepiece", sentencepiece.__version__)

In [ ]:
# Add-ons -> Secrets -> ajouter HF_TOKEN
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets")
except Exception as exc:
    print("HF token not loaded:", exc)

In [ ]:
!python ml/scripts/01_download.py

In [ ]:
!python ml/scripts/02_prepare_data.py

In [ ]:
!python ml/scripts/03_add_dua_token.py

In [ ]:
# Optionnel et long : paires FR synthétiques via M2M100 EN->FR.
# Utile pour la direction FR->DUA de l'extension bancaire.
!python ml/scripts/04_make_synthetic_fr.py

In [ ]:
# Smoke test : 2 000 exemples, 200 steps, sans synthétique.
!python ml/scripts/05_train.py --sample 2000 --max-steps 200 --no-synthetic

In [ ]:
# Run complet avec sauvegarde sur HF Hub.
# !python ml/scripts/05_train.py --push-to-hub <user>/edima-m2m100-dua

In [ ]:
!python ml/scripts/06_evaluate.py

In [ ]:
!python ml/scripts/07_export_ct2.py

## Après le run

- Télécharger `models/m2m100_dua_ct2/` (ou le récupérer depuis le HF Hub) pour le sidecar local.
- Les checkpoints sont dans `out/m2m100_dua/` ; `best/` contient le meilleur chrF.
- Étape 2 (adaptation domaine) : LoRA sur 200–500 paires bancaires or, après validation du glossaire.